# SPARQL Queries on ARIADNE Knowledge Base for ATRIUM Metacat

### Authors: Alessia Bardi (CNR-ISTI)

## Intro
These are the SPARQL queries for T3.2 of the ATRIUM project.
Metacat requires information about:
* Resource type
* Format
* Discipline
* Source
* Source-2 (the „source of the source“, if available; for example, if you harvest from an aggregator and you know the name of the original repository from which the aggregator itself harvested)
* Subjects

## Mapping ARIADNE KB to Metacat

TODO

In [1]:
#install libraries
!pip install rdflib
!pip install SPARQLWrapper
!pip install prettytable

  Using cached prettytable-3.11.0-py3-none-any.whl (28 kB)


In [2]:
#imports
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint

### Resource Types --> ARIADNE Subjects

In [8]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?type (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_ARIADNE_subject ?as .
    ?as skos:prefLabel ?type
}
GROUP BY ?type
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Resource Type', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['type']['value'], result['cnt']['value']])

print(t)



+---------------------+---------+
|    Resource Type    |  Count  |
+---------------------+---------+
|       Artefact      |  903100 |
|     Inscription     |  83375  |
|  Fieldwork archive  |  20600  |
| Scientific analysis |   9031  |
|        Burial       |  12198  |
|         Date        |  12833  |
|     Not provided    |    29   |
|    Site/monument    | 1347048 |
|      Fieldwork      |  292652 |
|       Rock art      |  29073  |
|       Maritime      |  100680 |
|   Fieldwork report  |  491946 |
|         Coin        |  474828 |
|   Building survey   |   1994  |
|    E-Publication    |    64   |
+---------------------+---------+


### Formats --> Data type

In [9]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?format (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_data_type ?dt .
    ?dt skos:prefLabel ?format
}
GROUP BY ?format
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Format', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['format']['value'], result['cnt']['value']])

print(t)



+-----------------+---------+
|      Format     |  Count  |
+-----------------+---------+
| Structured Data | 1510721 |
|    Geospatial   |   2004  |
|   Still Image   | 1011510 |
|       Text      |  104944 |
|       CAD       |   1322  |
|     Numeric     |   338   |
|     Software    |    4    |
|        3D       |   131   |
|      Video      |    64   |
|      Other      |    53   |
|      Audio      |    4    |
+-----------------+---------+


No resources with aocat:has_data_format, which is supposed to include the mime type (query below)

In [15]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?dt (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_data_format ?dt .
    
}
GROUP BY ?dt
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Format', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['dt']['value'], result['cnt']['value']])

print(t)

+--------+-------+
| Format | Count |
+--------+-------+
+--------+-------+


### Disciplines: in case of ARIADNE we have one discipline by default: Archaeology

### Source: the name of the data source from which ARIADNE collected record

In [18]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?source (count(?resource) AS ?cnt)  WHERE {
    { 
         GRAPH <https://ariadne-infrastructure.eu/datasourceApis> {
         ?g <http://www.d-net.research-infrastructures.eu/provenance/isApiOf> ?source
        } .
        GRAPH ?g {
        {?resource rdf:type aocat:AO_Collection} UNION {?resource rdf:type aocat:AO_Individual_Data_Resource}
        }
    }
}
GROUP BY ?source
''')
                
sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Format', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['source']['value'], result['cnt']['value']])

print(t)

+------------------------------------------------------------------+---------+
|                              Format                              |  Count  |
+------------------------------------------------------------------+---------+
|                     Archaeology Data Service                     | 1289258 |
|                             NIAM-BAS                             |   1952  |
|             Paliambela Kolindros Excavation Archive              |    3    |
|                               SND                                |  25873  |
|                          ZRC SAZU Arkas                          |   5718  |
|                          ZRC SAZU Zbiva                          |   3195  |
|            Magyar Nemzeti Múzeum Régészeti Adatbázis             |  63019  |
|                           FastiOnline                            |  14360  |
|                            DANS-EASY                             |  210468 |
| ROAD, The Role of Culture in Early Expansions of H

### Source 2: the source of the source. We may have something closer to this concept in the publishe field.

In [20]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?publisherName (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_publisher ?pub .
    ?pub aocat:has_name ?publisherName
    
}
GROUP BY ?publisherName
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Publisher Name', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['publisherName']['value'], result['cnt']['value']])

print(t)

+-------------------------------------------------------------------------------------------------------+---------+
|                                             Publisher Name                                            |  Count  |
+-------------------------------------------------------------------------------------------------------+---------+
|                                        Archaeology Data Service                                       | 1148067 |
|                                        Archaeology Data Service                                       | 1148067 |
|              National Institute of Archaeology with Museum, Bulgarian Academy of Sciences             |   1952  |
|                                          University of Patras                                         |    3    |
|                                     Swedish National Data Service                                     |   484   |
|                                                ZRC SAZU               

### Subjects --> Getty AAT subjects

Because we have a lot we report about the top 10. If we need to extend it is easy to do it changing the query.
The same Getty AAT may appear twice because 

In [23]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?gettyAAT ?subjectLabel (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_derived_subject ?gettyAAT .
    ?gettyAAT skos:prefLabel ?subjectLabel .
    }
GROUP BY ?gettyAAT ?subjectLabel
ORDER BY DESC(?cnt)
LIMIT 10
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Getty AAT URI', 'Label', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['gettyAAT']['value'], result['subjectLabel']['value'], result['cnt']['value']])

print(t)

+--------------------------------------+------------------------------------------+--------+
|            Getty AAT URI             |                  Label                   | Count  |
+--------------------------------------+------------------------------------------+--------+
| http://vocab.getty.edu/aat/300194572 |        EARLY WESTERN WORLD COINS         | 230616 |
| http://vocab.getty.edu/aat/300194572 |        Early Western World coins         | 230616 |
| http://vocab.getty.edu/aat/300008022 |                 trenches                 | 167655 |
| http://vocab.getty.edu/aat/300444153 | settlements (sites of small communities) | 165581 |
| http://vocab.getty.edu/aat/300005433 |                  houses                  | 158185 |
| http://vocab.getty.edu/aat/300005433 |                  Houses                  | 158185 |
| http://vocab.getty.edu/aat/300008027 |            pits (earthworks)             | 146639 |
| http://vocab.getty.edu/aat/300008027 |                   pits       

Alternative: this query takes more time to run but it should not have duplicate labels. When I tried it took 2 minutes 18 seconds.

In [25]:
from rdflib import Graph
from SPARQLWrapper import SPARQLWrapper, JSON, N3
from pprint import pprint
from prettytable import PrettyTable


sparql = SPARQLWrapper('https://ariadne-graphdb.cloud.d4science.org/repositories/ariadneplus-pr01')
sparql.setQuery('''
PREFIX aocat: <https://www.ariadne-infrastructure.eu/resource/ao/cat/1.1/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX skosxl: <http://www.w3.org/2008/05/skos-xl#>
SELECT ?gettyAAT ?subjectLabel (count(?resource) AS ?cnt) WHERE {
    ?resource aocat:has_derived_subject ?gettyAAT .
    GRAPH <http://vocab.getty.edu/aat/> {
        ?gettyAAT skosxl:prefLabel ?label .
        ?label skosxl:literalForm ?subjectLabel .
        FILTER(LANG(?subjectLabel) = 'en')
    }
}
GROUP BY ?gettyAAT ?subjectLabel
ORDER BY DESC(?cnt)
LIMIT 10
''')

sparql.setReturnFormat(JSON)
qres = sparql.query().convert()

t = PrettyTable(['Getty AAT URI', 'Label', 'Count'])

for result in qres['results']['bindings']:
    t.add_row([result['gettyAAT']['value'], result['subjectLabel']['value'], result['cnt']['value']])

print(t)

+--------------------------------------+------------------------------------------+--------+
|            Getty AAT URI             |                  Label                   | Count  |
+--------------------------------------+------------------------------------------+--------+
| http://vocab.getty.edu/aat/300194572 |        Early Western World coins         | 230616 |
| http://vocab.getty.edu/aat/300008022 |                 trenches                 | 167655 |
| http://vocab.getty.edu/aat/300444153 | settlements (sites of small communities) | 165581 |
| http://vocab.getty.edu/aat/300005433 |                  houses                  | 158185 |
| http://vocab.getty.edu/aat/300008027 |            pits (earthworks)             | 146639 |
| http://vocab.getty.edu/aat/300193015 |           vessels (containers)           | 121036 |
| http://vocab.getty.edu/aat/300027267 |                 reports                  | 104918 |
| http://vocab.getty.edu/aat/300410475 |     settlement (population ac